# 01 — Europarl English-German data pipeline

Acquire, verify, extract, shard, manifest, and load Europarl v7 English-German training data plus `newstest2013` development data.

| Property | Value |
| --- | --- |
| Status | Revised for the maintainer-approved Europarl-only training campaign |
| Mapped issue | [#2](https://github.com/majorgilles/transformer-2017-reproduction/issues/2) |
| Depends on | `00_environment_contract.ipynb` / issue #1 |
| Final test | `newstest2014` remains unopened |


In [1]:
#| default_exp data


## Shared content-identity primitives

SHA-256 identifies immutable content across data shards, tokenizers, and later checkpoints. Notebook 01 owns the implementation while `identity.py` provides a domain-neutral import boundary.

In [2]:
#| exporti identity
import hashlib
from pathlib import Path as IdentityPath

__all__ = ["sha256_bytes", "sha256_file"]


def sha256_bytes(content: bytes) -> str:
    """Return the lowercase SHA-256 digest of bytes."""

    return hashlib.sha256(content).hexdigest()


def sha256_file(path: IdentityPath) -> str:
    """Return the lowercase SHA-256 digest of a file without loading it whole."""

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

### Visible result: domain-neutral content identity

A fixed byte fixture demonstrates that later modules receive the same deterministic identity without depending on the data module.

In [3]:
identity_fixture = b"transformer-2017-reproduction"
identity_fixture_digest = sha256_bytes(identity_fixture)

assert identity_fixture_digest == (
    "77fe8d526ebc755d856904ac3d95d79b696a495fc42b580b89f2424827ea95b6"
)

print("Shared content identity:")
print(f"  byte count: {len(identity_fixture)}")
print(f"  SHA-256: {identity_fixture_digest}")

Shared content identity:
  byte count: 29
  SHA-256: 77fe8d526ebc755d856904ac3d95d79b696a495fc42b580b89f2424827ea95b6


## Contract and sources

This notebook owns the complete reusable data-loading boundary required by tokenization and training. The active training corpus is Europarl v7 only; `newstest2013` remains the development set. Common Crawl and News Commentary are excluded so corpus-domain changes cannot dominate the training-loss diagnostic.

- Vaswani et al., [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), §5.1: WMT14 English-German is the paper task.
- [Official WMT14 translation task](https://statmt.org/wmt14/translation-task.html): archives and development-set policy.
- [ADR 0006](../docs/adr/0006-europarl-only-training-control.md): rationale for the Europarl-only deviation.
- [`docs/fidelity-matrix.md`](../docs/fidelity-matrix.md): corpus and final-test freeze contracts.

The maintainer approved this source restriction in chat. No text, archive, or generated shard is committed; all live under ignored `data/`.


## Exported contracts and pinned source specification


In [4]:
#| export

import argparse
import json
import shutil
import tarfile
import time
from collections.abc import Iterator, Sequence
from itertools import zip_longest
from pathlib import Path
from typing import Final, Literal, cast
from urllib.error import URLError
from urllib.request import Request, urlopen

from pydantic import BaseModel, ConfigDict

from transformer_2017_reproduction.identity import (
    sha256_bytes as _sha256_bytes,
)
from transformer_2017_reproduction.identity import (
    sha256_file as _sha256_file,
)

Split = Literal["train", "development"]


class ParallelExample(BaseModel):
    """One aligned English source and German target sentence."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    source_text: str
    target_text: str


class CorpusSource(BaseModel):
    """Pinned WMT archive and paired members consumed by the pipeline."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    name: str
    split: Split
    url: str
    archive_filename: str
    source_member: str
    target_member: str


class AcquisitionRecord(BaseModel):
    """Content identity of one downloaded source archive."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    name: str
    url: str
    relative_path: str
    sha256: str
    byte_count: int


class SourceProcessingRecord(BaseModel):
    """Deterministic processing counts for one parallel source."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    name: str
    split: Split
    input_pairs: int
    emitted_pairs: int
    skipped_empty_pairs: int


class ShardRecord(BaseModel):
    """Identity and summary of one immutable JSON Lines shard."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    split: Split
    source_name: str
    relative_path: str
    sha256: str
    example_count: int
    byte_count: int


class DatasetManifest(BaseModel):
    """Complete identity of the acquired and sharded WMT14 dataset."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    schema_version: int
    dataset_name: str
    source_language: str
    target_language: str
    shard_size: int
    acquisitions: tuple[AcquisitionRecord, ...]
    sources: tuple[SourceProcessingRecord, ...]
    shards: tuple[ShardRecord, ...]
    total_examples: int


WMT14_EN_DE_SOURCES: Final[tuple[CorpusSource, ...]] = (
    CorpusSource(
        name="europarl-v7",
        split="train",
        url="https://statmt.org/wmt13/training-parallel-europarl-v7.tgz",
        archive_filename="training-parallel-europarl-v7.tgz",
        source_member="training/europarl-v7.de-en.en",
        target_member="training/europarl-v7.de-en.de",
    ),
    CorpusSource(
        name="newstest2013",
        split="development",
        url="https://statmt.org/wmt14/dev.tgz",
        archive_filename="dev.tgz",
        source_member="dev/newstest2013.en",
        target_member="dev/newstest2013.de",
    ),
)

## Acquisition, sharding, manifests, and loading


In [5]:
#| export
def serialize_examples(examples: Sequence[ParallelExample]) -> bytes:
    """Serialize examples as deterministic UTF-8 JSON Lines bytes."""

    return b"".join((example.model_dump_json() + "\n").encode("utf-8") for example in examples)


def _download_once(url: str, destination: Path) -> None:
    part_path = destination.with_suffix(destination.suffix + ".part")
    offset = part_path.stat().st_size if part_path.exists() else 0
    headers = {"User-Agent": "transformer-2017-reproduction/0.1"}
    if offset:
        headers["Range"] = f"bytes={offset}-"

    request = Request(url, headers=headers)
    with urlopen(request, timeout=120) as response:
        status = response.getcode()
        append = offset > 0 and status == 206
        mode = "ab" if append else "wb"
        with part_path.open(mode) as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)
    part_path.replace(destination)


def download_archive(source: CorpusSource, archive_dir: Path) -> AcquisitionRecord:
    """Download one archive with retry/resume and return its content identity."""

    archive_dir.mkdir(parents=True, exist_ok=True)
    destination = archive_dir / source.archive_filename
    if not destination.exists():
        last_error: OSError | URLError | None = None
        for attempt in range(1, 6):
            try:
                _download_once(source.url, destination)
                break
            except (OSError, URLError) as error:
                last_error = error
                if attempt == 5:
                    raise RuntimeError(f"failed to download {source.url}") from last_error
                time.sleep(float(attempt))

    return AcquisitionRecord(
        name=source.name,
        url=source.url,
        relative_path=destination.as_posix(),
        sha256=_sha256_file(destination),
        byte_count=destination.stat().st_size,
    )


def _copy_archive_member(archive: tarfile.TarFile, member_name: str, destination: Path) -> None:
    if destination.exists():
        return
    member = archive.getmember(member_name)
    extracted = archive.extractfile(member)
    if extracted is None:
        raise FileNotFoundError(f"archive member is not a file: {member_name}")

    destination.parent.mkdir(parents=True, exist_ok=True)
    part_path = destination.with_suffix(destination.suffix + ".part")
    with extracted, part_path.open("wb") as output:
        shutil.copyfileobj(extracted, output, length=1024 * 1024)
    part_path.replace(destination)


def extract_parallel_source(
    source: CorpusSource,
    archive_path: Path,
    extracted_dir: Path,
) -> tuple[Path, Path]:
    """Extract only the English and German members needed from an archive."""

    source_path = extracted_dir / f"{source.name}.en"
    target_path = extracted_dir / f"{source.name}.de"
    if source_path.exists() and target_path.exists():
        return source_path, target_path

    with tarfile.open(archive_path, mode="r:gz") as archive:
        _copy_archive_member(archive, source.source_member, source_path)
        _copy_archive_member(archive, source.target_member, target_path)
    return source_path, target_path


def _write_immutable_shard(
    root: Path,
    relative_path: str,
    *,
    split: Split,
    source_name: str,
    examples: Sequence[ParallelExample],
) -> ShardRecord:
    content = serialize_examples(examples)
    shard_path = root / relative_path
    shard_path.parent.mkdir(parents=True, exist_ok=True)
    if shard_path.exists():
        if shard_path.read_bytes() != content:
            raise FileExistsError(f"refusing to replace immutable shard: {shard_path}")
    else:
        part_path = shard_path.with_suffix(shard_path.suffix + ".part")
        part_path.write_bytes(content)
        part_path.replace(shard_path)

    return ShardRecord(
        split=split,
        source_name=source_name,
        relative_path=relative_path,
        sha256=_sha256_bytes(content),
        example_count=len(examples),
        byte_count=len(content),
    )


def shard_parallel_source(
    root: Path,
    source: CorpusSource,
    source_path: Path,
    target_path: Path,
    shard_size: int,
) -> tuple[tuple[ShardRecord, ...], SourceProcessingRecord]:
    """Stream aligned files into deterministic immutable shards."""

    if shard_size < 1:
        raise ValueError("shard_size must be at least 1")

    records: list[ShardRecord] = []
    batch: list[ParallelExample] = []
    input_pairs = 0
    emitted_pairs = 0
    skipped_empty_pairs = 0
    shard_index = 0
    shard_root = f"processed/shards-{shard_size}/{source.split}"

    with (
        source_path.open("r", encoding="utf-8", newline="\n") as source_handle,
        target_path.open("r", encoding="utf-8", newline="\n") as target_handle,
    ):
        for source_line, target_line in zip_longest(source_handle, target_handle):
            if source_line is None or target_line is None:
                raise ValueError(f"unaligned line counts for {source.name}")
            input_pairs += 1
            source_text = source_line.rstrip("\r\n")
            target_text = target_line.rstrip("\r\n")
            if not source_text or not target_text:
                skipped_empty_pairs += 1
                continue

            batch.append(
                ParallelExample(
                    source_text=source_text,
                    target_text=target_text,
                )
            )
            emitted_pairs += 1
            if len(batch) == shard_size:
                relative_path = f"{shard_root}/{source.name}-{shard_index:05d}.jsonl"
                records.append(
                    _write_immutable_shard(
                        root,
                        relative_path,
                        split=source.split,
                        source_name=source.name,
                        examples=batch,
                    )
                )
                batch = []
                shard_index += 1

    if batch:
        relative_path = f"{shard_root}/{source.name}-{shard_index:05d}.jsonl"
        records.append(
            _write_immutable_shard(
                root,
                relative_path,
                split=source.split,
                source_name=source.name,
                examples=batch,
            )
        )

    processing = SourceProcessingRecord(
        name=source.name,
        split=source.split,
        input_pairs=input_pairs,
        emitted_pairs=emitted_pairs,
        skipped_empty_pairs=skipped_empty_pairs,
    )
    return tuple(records), processing


def manifest_as_bytes(manifest: DatasetManifest) -> bytes:
    """Serialize a manifest deterministically for storage and hashing."""

    return (manifest.model_dump_json(indent=2) + "\n").encode("utf-8")


def write_manifest(root: Path, manifest: DatasetManifest) -> Path:
    """Write the deterministic manifest, refusing an identity-changing replacement."""

    path = root / "manifests" / f"{manifest.dataset_name}.json"
    content = manifest_as_bytes(manifest)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"refusing to replace manifest with different content: {path}")
    if not path.exists():
        part_path = path.with_suffix(path.suffix + ".part")
        part_path.write_bytes(content)
        part_path.replace(path)
    return path


def load_manifest(path: Path) -> DatasetManifest:
    """Load and validate a dataset manifest from disk."""

    return DatasetManifest.model_validate_json(path.read_bytes())


def prepare_europarl_en_de(root: Path, shard_size: int = 100_000) -> DatasetManifest:
    """Acquire and shard Europarl training plus newstest2013 development data."""

    archive_dir = root / "raw" / "archives"
    extracted_dir = root / "raw" / "parallel"
    acquisitions: list[AcquisitionRecord] = []
    source_records: list[SourceProcessingRecord] = []
    shard_records: list[ShardRecord] = []

    for source in WMT14_EN_DE_SOURCES:
        acquisition = download_archive(source, archive_dir)
        archive_path = archive_dir / source.archive_filename
        acquisitions.append(
            acquisition.model_copy(
                update={"relative_path": archive_path.relative_to(root).as_posix()}
            )
        )
        source_path, target_path = extract_parallel_source(
            source,
            archive_path,
            extracted_dir,
        )
        records, processing = shard_parallel_source(
            root,
            source,
            source_path,
            target_path,
            shard_size,
        )
        shard_records.extend(records)
        source_records.append(processing)

    manifest = DatasetManifest(
        schema_version=1,
        dataset_name=f"europarl-en-de-shard-{shard_size}",
        source_language="en",
        target_language="de",
        shard_size=shard_size,
        acquisitions=tuple(acquisitions),
        sources=tuple(source_records),
        shards=tuple(shard_records),
        total_examples=sum(record.emitted_pairs for record in source_records),
    )
    write_manifest(root, manifest)
    return manifest


def load_shard(root: Path, record: ShardRecord) -> tuple[ParallelExample, ...]:
    """Verify and load one shard."""

    path = root / record.relative_path
    content = path.read_bytes()
    if len(content) != record.byte_count:
        raise ValueError(f"byte count mismatch for {record.relative_path}")
    if _sha256_bytes(content) != record.sha256:
        raise ValueError(f"SHA-256 mismatch for {record.relative_path}")
    examples = tuple(ParallelExample.model_validate_json(line) for line in content.splitlines())
    if len(examples) != record.example_count:
        raise ValueError(f"example count mismatch for {record.relative_path}")
    return examples


def iter_manifest_examples(
    root: Path,
    manifest: DatasetManifest,
    split: Split,
) -> Iterator[ParallelExample]:
    """Yield verified examples one shard at a time for a selected split."""

    for record in manifest.shards:
        if record.split == split:
            yield from load_shard(root, record)


def iter_bpe_training_text(root: Path, manifest: DatasetManifest) -> Iterator[str]:
    """Yield source then target text for shared BPE training."""

    for example in iter_manifest_examples(root, manifest, "train"):
        yield example.source_text
        yield example.target_text

## Exported acquisition CLI


In [6]:
#| export
def main(argv: Sequence[str] | None = None) -> int:
    """Run the Europarl English-German acquisition and sharding pipeline."""

    parser = argparse.ArgumentParser(description=main.__doc__)
    parser.add_argument("--root", default="data/europarl_en_de")
    parser.add_argument("--shard-size", type=int, default=100_000)
    args = parser.parse_args(argv)
    root = Path(cast(str, args.root))
    shard_size = cast(int, args.shard_size)
    manifest = prepare_europarl_en_de(root, shard_size)
    manifest_path = root / "manifests" / f"{manifest.dataset_name}.json"
    report = {
        "manifest": manifest_path.as_posix(),
        "manifest_sha256": _sha256_file(manifest_path),
        "shards": len(manifest.shards),
        "total_examples": manifest.total_examples,
        "train_examples": sum(
            record.example_count for record in manifest.shards if record.split == "train"
        ),
        "development_examples": sum(
            record.example_count for record in manifest.shards if record.split == "development"
        ),
    }
    print(json.dumps(report, indent=2, sort_keys=True))
    return 0

## Focused synthetic end-to-end test


In [7]:
from tempfile import TemporaryDirectory

fixture = (
    ParallelExample(source_text="The lamp is on.", target_text="Die Lampe ist an."),
    ParallelExample(source_text="We open the window.", target_text="Wir öffnen das Fenster."),
    ParallelExample(source_text="The book is here.", target_text="Das Buch ist hier."),
)

with TemporaryDirectory() as temporary_directory:
    test_root = Path(temporary_directory)
    source_path = test_root / "fixture.en"
    target_path = test_root / "fixture.de"
    source_path.write_text(
        "".join(f"{example.source_text}\n" for example in fixture),
        encoding="utf-8",
    )
    target_path.write_text(
        "".join(f"{example.target_text}\n" for example in fixture),
        encoding="utf-8",
    )
    fixture_source = CorpusSource(
        name="fixture",
        split="train",
        url="https://example.invalid/fixture.tgz",
        archive_filename="fixture.tgz",
        source_member="fixture.en",
        target_member="fixture.de",
    )
    fixture_shards, fixture_processing = shard_parallel_source(
        test_root,
        fixture_source,
        source_path,
        target_path,
        shard_size=2,
    )
    fixture_manifest = DatasetManifest(
        schema_version=1,
        dataset_name="fixture",
        source_language="en",
        target_language="de",
        shard_size=2,
        acquisitions=(),
        sources=(fixture_processing,),
        shards=fixture_shards,
        total_examples=3,
    )
    fixture_manifest_path = write_manifest(test_root, fixture_manifest)
    reloaded_manifest = load_manifest(fixture_manifest_path)
    reloaded_examples = tuple(iter_manifest_examples(test_root, reloaded_manifest, "train"))
    bpe_text = tuple(iter_bpe_training_text(test_root, reloaded_manifest))

    first_record = reloaded_manifest.shards[0]
    first_shard_path = test_root / first_record.relative_path
    original_content = first_shard_path.read_bytes()
    first_shard_path.write_bytes(original_content + b"corruption")
    try:
        load_shard(test_root, first_record)
    except ValueError:
        pass
    else:
        raise AssertionError("corrupted shard was accepted")
    first_shard_path.write_bytes(original_content)

    try:
        _write_immutable_shard(
            test_root,
            first_record.relative_path,
            split="train",
            source_name="fixture",
            examples=(ParallelExample(source_text="different", target_text="anders"),),
        )
    except FileExistsError:
        pass
    else:
        raise AssertionError("immutable shard was replaced")

assert reloaded_examples == fixture
assert tuple(record.example_count for record in fixture_shards) == (2, 1)
assert bpe_text == tuple(
    text for example in fixture for text in (example.source_text, example.target_text)
)
print(fixture_manifest.model_dump_json(indent=2))

{
  "schema_version": 1,
  "dataset_name": "fixture",
  "source_language": "en",
  "target_language": "de",
  "shard_size": 2,
  "acquisitions": [],
  "sources": [
    {
      "name": "fixture",
      "split": "train",
      "input_pairs": 3,
      "emitted_pairs": 3,
      "skipped_empty_pairs": 0
    }
  ],
  "shards": [
    {
      "split": "train",
      "source_name": "fixture",
      "relative_path": "processed/shards-2/train/fixture-00000.jsonl",
      "sha256": "f6fd4d3ec1668d2b7d2fed55504c2be6f9934e0b75c22ba192fe26304535a459",
      "example_count": 2,
      "byte_count": 147
    },
    {
      "split": "train",
      "source_name": "fixture",
      "relative_path": "processed/shards-2/train/fixture-00001.jsonl",
      "sha256": "12876823177cf2d19487488eaecf1e2e728b1dc3fdffa3779af0fe21964fc9d8",
      "example_count": 1,
      "byte_count": 71
    }
  ],
  "total_examples": 3
}


## Canonical local Europarl artifacts

Command:

```powershell
uv run transformer-data --root data/europarl_en_de --shard-size 100000
```

| Identity | Value |
| --- | --- |
| Manifest | `data/europarl_en_de/manifests/europarl-en-de-shard-100000.json` |
| Manifest SHA-256 | `1c74b35bdb910d0e328b111bb08b631f6a88c64edb631baacf2ff39999b88281` |
| Training examples | 1,908,920 |
| Development examples | 3,000 |
| Shards | 20 train + 1 development |
| Processed shard storage | 0.634 GiB |
| Final-test matches on disk | 0 |

| Archive | SHA-256 |
| --- | --- |
| Europarl v7 | `0224c7c710c8a063dfd893b0cc0830202d61f4c75c17eb8e31836103d27d96e7` |
| `dev.tgz` | `cda0f85309e8ea4c9c2bc142cd795fb2771a5939ed5b4527b525dae05fa0c145` |

A full exported-loader pass verifies 1,908,920 training examples and 3,000 development examples against every recorded byte count and shard SHA-256. Notebook 02 consumes `iter_bpe_training_text`. `newstest2014` was not downloaded.


## Explicitly deferred

Shared BPE training, model tensors, optimization, checkpoints, and final-test acquisition. Europarl train and `newstest2013` development acquisition and loading are owned here.


## HITL checkpoint

The maintainer approved restricting active training to Europarl v7 in chat on 2026-09-04. Approval retains `newstest2013` for development, the 100,000-example shard size, immutable identities, and the `newstest2014` freeze.
